In [ ]:
import sys
from pathlib import Path

# Make the mobility package importable when running from notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from mobility.config import MobilityConfig
from mobility import geometry, transitions, stats

DATA_DIR = REPO_ROOT / "data"
OUTPUTS_DIR = REPO_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root :", REPO_ROOT)
print("Data dir  :", DATA_DIR)

In [ ]:
cfg = MobilityConfig()
np.random.seed(cfg.seed)
print(cfg)

In [ ]:
df_occ = geometry.load_occupations(DATA_DIR / "occupation_embeddings_polar_scaled.csv")
df_tasks = geometry.load_tasks(DATA_DIR / "task_embeddings_polar_scaled.csv")

print(f"Occupations (O*NET-detailed): {len(df_occ)}")
print(f"Tasks (O*NET-detailed)      : {len(df_tasks)}")

df_occ.head(3)

In [ ]:
df_soc_centres = geometry.build_soc_centres(df_occ)
df_task_radii = geometry.build_task_radii(df_occ, df_tasks, task_policy=cfg.task_policy)

print(f"SOC2018 centres   : {len(df_soc_centres)}")
print(f"SOC2018 with radii: {len(df_task_radii)}")

df_soc_centres.head(3)

In [ ]:
df_ipums = transitions.load_ipums_transitions(DATA_DIR / "transitions_20_24.csv")
F, P, soc_index = transitions.build_transition_matrix(df_ipums)

print(f"IPUMS rows used : {len(df_ipums)}")
print(f"SOC universe (N): {len(soc_index)}")
print(f"Nonzero flows   : {int(np.count_nonzero(F))}")
print(f"Rows with sum>0 : {int(np.sum(F.sum(axis=1) > 0))}")

In [ ]:
df_edges = transitions.build_edges(
    P=P,
    soc_index=soc_index,
    df_soc_centres=df_soc_centres,
    df_task_radii=df_task_radii,
    p_threshold=cfg.p_threshold,
    top_k=cfg.top_k,
)

print(f"Edges total    : {len(df_edges)}")
print(f"Sources covered: {df_edges['src_soc2018'].nunique()}")
print(f"Targets covered: {df_edges['tgt_soc2018'].nunique()}")

df_edges.head(5)

In [ ]:
# Quick weighted summaries to compare against the old notebook's KPIs.
w = df_edges["wP"].to_numpy(float)
d = df_edges["d_xy"].to_numpy(float)
u_R = df_edges["u_R"].to_numpy(float)

summary = pd.DataFrame([{
    "n_edges": len(df_edges),
    "w_sum": float(w.sum()),
    "d_median_w": stats.wquantile(d, w, 0.50),
    "d_p75_w":    stats.wquantile(d, w, 0.75),
    "d_p90_w":    stats.wquantile(d, w, 0.90),
    "u_R_median_w": stats.wquantile(u_R, w, 0.50),
}])
summary

In [ ]:
from mobility import overlap

df_edges_full = overlap.merge_target_radii(df_edges, df_task_radii)
df_edges_full = overlap.add_signed_overlap(df_edges_full)

print(f"Edges with overlap: {df_edges_full['overlap_signed'].notna().sum()}")
print(f"Share with positive overlap (unweighted): "
      f"{(df_edges_full['overlap_signed'] > 0).mean():.3f}")

overlap.overlap_summary(df_edges_full)

In [ ]:
w = df_edges_full["wP"].to_numpy(float)
d = df_edges_full["d_xy"].to_numpy(float)
u_R = df_edges_full["u_R"].to_numpy(float)

xs_uR, cdf_uR = stats.weighted_ecdf(u_R, w)
thresholds = [0.10, 0.25, 0.33, 0.50, 1.00, 1.50, 2.00]

kpi_table = pd.DataFrame([{
    "n_edges": len(df_edges_full),
    "d_median": stats.wquantile(d, w, 0.50),
    "d_p75":    stats.wquantile(d, w, 0.75),
    "d_p90":    stats.wquantile(d, w, 0.90),
    "u_R_median": stats.wquantile(u_R, w, 0.50),
    "u_R_p75":    stats.wquantile(u_R, w, 0.75),
    "u_R_p90":    stats.wquantile(u_R, w, 0.90),
    **{f"share_u_R_le_{t:.2f}".replace(".", "p"):
       100.0 * stats.weighted_cdf_at(xs_uR, cdf_uR, t)
       for t in thresholds},
}])

kpi_table.to_csv(TABLES_DIR / "section_3_1_1_kpis.csv", index=False)
kpi_table.T

In [ ]:
from mobility import plots

plots.plot_ecdf_absolute(
    df_edges_full,
    FIGURES_DIR / "ecdf_hoplength_abs_weighted.pdf",
)
plots.plot_ecdf_normalised(
    df_edges_full,
    FIGURES_DIR / "ecdf_hoplength_norm_weighted_uR.pdf",
)
plots.plot_overlap_vs_hop(
    df_edges_full,
    FIGURES_DIR / "overlap_vs_hop_uR.pdf",
)
plots.plot_overlap_cdf(
    df_edges_full,
    FIGURES_DIR / "overlap_cdf_uR.pdf",
)

print("Figures written to:", FIGURES_DIR)
for fp in sorted(FIGURES_DIR.glob("*.pdf")):
    print(f"  {fp.name}")

In [ ]:
from mobility import fields as mfields

# Use full edge table (with target radii for completeness; not strictly needed
# for fields but harmless).
edges_arrays = mfields.prepare_edges(df_edges_full)

field_cfg = mfields.FieldConfig(
    bandwidth=0.08,
    n_grid=120,
    batch=1024,
)
grid = mfields.make_grid(field_cfg)

cache_dir = OUTPUTS_DIR / "field_cache"

# Global absolute field
field_global = mfields.compute_or_load(
    name="global_absolute",
    edges=edges_arrays, grid=grid, cfg=field_cfg,
    cache_dir=cache_dir,
)
print(f"Global field: n_edges={field_global.n_edges}, "
      f"R median={np.nanmedian(field_global.R):.3f}")

# Global drift and residual
drift = mfields.global_drift(edges_arrays)
print(f"Global drift: u={drift[0]:.4f}, v={drift[1]:.4f}, "
      f"angle={np.degrees(np.arctan2(drift[1], drift[0])):.1f}°")

dx_res, dy_res, _ = mfields.residual_directions(edges_arrays, drift)
field_residual_global = mfields.compute_or_load(
    name="global_residual",
    edges=edges_arrays, grid=grid, cfg=field_cfg,
    cache_dir=cache_dir,
    direction_x=dx_res, direction_y=dy_res,
    drift=drift,
)
print(f"Residual field: R median={np.nanmedian(field_residual_global.R):.3f}")

In [ ]:
from mobility import systems as msys

best, all_runs = msys.detect_systems(
    df_edges_full,
    n_components=2,
    n_runs=1,
    n_init=5,
    min_system_frac=0.15,
    select_by="flow_between",
    base_seed=0,
)

print(f"Selected seed   : {best.seed}")
print(f"Within systems  : {best.flow_within:.1f}%")
print(f"Crossing systems: {best.flow_between:.1f}%")
print(f"Silhouette      : {best.silhouette:.3f}")
print(f"Smallest system : {best.min_sys_frac:.1f}% of total weight")

print("\nFlow matrix (% of total weight):")
print(pd.DataFrame(
    best.flow_matrix.round(1),
    index=[f"From sys {k}" for k in range(best.n_components)],
    columns=[f"To sys {k}" for k in range(best.n_components)],
))

In [ ]:
src_sys, tgt_sys, crossing = msys.assign_edge_systems(
    df_edges_full, best.soc_system,
)

# Per-system fields: edges where both endpoints are in system k
field_systems = {}
for k in range(best.n_components):
    mask_k = (src_sys == k) & (tgt_sys == k)
    field_systems[k] = mfields.compute_or_load(
        name=f"system_{k}",
        edges=edges_arrays, grid=grid, cfg=field_cfg,
        cache_dir=cache_dir,
        edge_mask=mask_k,
    )
    print(f"System {k}: {int(mask_k.sum())} within-edges")

# Crossing field
field_crossing = mfields.compute_or_load(
    name="crossing",
    edges=edges_arrays, grid=grid, cfg=field_cfg,
    cache_dir=cache_dir,
    edge_mask=crossing,
)
print(f"Crossing  : {int(crossing.sum())} edges")

# Attractor poles and dynamic separatrix
poles = msys.attractor_poles(df_edges_full, best.soc_system,
                              best.n_components)
print(f"\nPoles:")
for k, p in enumerate(poles):
    angle = np.degrees(np.arctan2(p[1], p[0])) % 360
    print(f"  Pole {k}: ({p[0]:+.3f}, {p[1]:+.3f})  angle={angle:.1f}°")

separatrix = msys.dynamic_separatrix(
    edges_arrays, grid, field_cfg, crossing, poles,
)

In [ ]:
from mobility import plots

# Background occupations: original O*NET-detailed positions
xo = df_occ["x"].to_numpy()
yo = df_occ["y"].to_numpy()

plots.plot_mobility_systems_combined(
    field_systems=field_systems,
    field_residual=field_crossing,
    field_global=field_global,
    grid=grid,
    separatrix=separatrix,
    poles=poles,
    occupations_xy=(xo, yo),
    out_path=FIGURES_DIR / "mobility_systems_combined.png",
    theme="paper",
)
print(f"Saved: {FIGURES_DIR / 'mobility_systems_combined.png'}")

In [ ]:
edge_subsets = msys.partition_edges_by_system(
    df_edges_full, best.soc_system, best.n_components,
)

print("Edge subset sizes:")
for key, sub in edge_subsets.items():
    print(f"  {key:12s}: {len(sub):5d} edges, w_sum={sub['wP'].sum():.2f}")

# Compute and print median normalised hop length per subset
print("\nMedian u_R per subset:")
for key, sub in edge_subsets.items():
    u = sub["u_R"].to_numpy(dtype=float)
    w = sub["wP"].to_numpy(dtype=float)
    print(f"  {key:12s}: median u_R = {stats.wquantile(u, w, 0.50):.3f}")

plots.plot_system_cdfs(
    edge_subsets=edge_subsets,
    out_path=FIGURES_DIR / "transition_length_cdf_normalised.png",
    theme="paper",
)
print(f"\nSaved: {FIGURES_DIR / 'transition_length_cdf_normalised.png'}")

In [ ]:
from mobility import directions as mdir

# Project displacement onto tangential direction at source
edge_subsets_proj = {}
for key, sub in edge_subsets.items():
    edge_subsets_proj[key] = mdir.compute_tangential_projection(
        sub, df_soc_centres, normalise=True,
    )

# Compute angular curves per subset
curves = {}
for key, sub in edge_subsets_proj.items():
    bin_centres, curve, se, weights = mdir.angular_curve(
        sub, n_bins=36, smooth_window=3,
    )
    curves[key] = (bin_centres, curve, se, weights)
    n_valid = int(np.isfinite(curve).sum())
    mean_proj = float(np.nanmean(curve))
    mean_se = float(np.nanmean(se))
    print(f"  {key:12s}: {n_valid}/36 bins, "
          f"mean proj_xi={mean_proj:+.3f}, mean SE={mean_se:.3f}")

# Plot without bands (default)
plots.plot_angular_directions(
    curves=curves,
    out_path=FIGURES_DIR / "mobility_directions_by_system.png",
    poles=poles,
    pole_labels=[f"Pole {i}" for i in range(best.n_components)],
    theme="paper",
    origin_deg=135.0,
)
print(f"Saved: {FIGURES_DIR / 'mobility_directions_by_system.png'}")

# Plot with bands (alternative version)
plots.plot_angular_directions(
    curves=curves,
    out_path=FIGURES_DIR / "mobility_directions_by_system_with_band.png",
    poles=poles,
    pole_labels=[f"Pole {i}" for i in range(best.n_components)],
    theme="paper",
    origin_deg=135.0,
    show_band=True,
    band_n_se=1.0,
    band_alpha=0.18,
)
print(f"Saved: {FIGURES_DIR / 'mobility_directions_by_system_with_band.png'}")

In [ ]:
plots.plot_gmm_diagnostic(
    df_edges=df_edges_full,
    soc_system=best.soc_system,
    poles=poles,
    grid=grid,
    separatrix=separatrix,
    occupations_xy=(xo, yo),
    out_path=FIGURES_DIR / "gmm_4d_systems.png",
    flow_within_pct=best.flow_within,
    flow_between_pct=best.flow_between,
    theme="presentation",
)
print(f"Saved: {FIGURES_DIR / 'gmm_4d_systems.png'}")

In [ ]:
plots.plot_mobility_field_combined(
    field_systems=field_systems,
    grid=grid,
    separatrix=separatrix,
    poles=poles,
    occupations_xy=(xo, yo),
    out_path=FIGURES_DIR / "mobility_field_combined.png",
    sys_labels={0: "System 0 (cognitive)", 1: "System 1 (physical)"},
    theme="presentation",
)
print(f"Saved: {FIGURES_DIR / 'mobility_field_combined.png'}")

plots.plot_mobility_field_residual(
    field_residual=field_crossing,
    grid=grid,
    separatrix=separatrix,
    poles=poles,
    occupations_xy=(xo, yo),
    out_path=FIGURES_DIR / "mobility_field_residual.png",
    theme="presentation",
)
print(f"Saved: {FIGURES_DIR / 'mobility_field_residual.png'}")

In [ ]:
from mobility import diagnostics as mdiag

# Build a SOC -> title map from the occupation table
title_map = (
    df_occ.dropna(subset=["soc2018", "Title"])
          .groupby("soc2018")["Title"]
          .first()
)

# Take crossing edges and add u_R for the summary table
df_edges_crossing = edge_subsets["crossing"].copy()

# Classify each crossing edge by manager-direction and segment
df_edges_classified = mdiag.classify_subgroups(
    df_edges_crossing=df_edges_crossing,
    df_soc_centres=df_soc_centres,
    title_map=title_map,
)

print("Subgroup edge counts:")
counts = df_edges_classified["subgroup"].value_counts()
for sg in mdiag.SUBGROUP_ORDER:
    if sg in counts.index:
        n = int(counts[sg])
        w = float(df_edges_classified.loc[
            df_edges_classified["subgroup"] == sg, "wP"
        ].sum())
        print(f"  {sg:30s}: n={n:5d}, w_sum={w:6.2f}")

# Source-position aggregation (used by the detailed plot)
src_positions = mdiag.source_positions_by_subgroup(
    df_edges_classified=df_edges_classified,
    df_soc_centres=df_soc_centres,
    title_map=title_map,
)

# Subgroup-level summary table (paper appendix material)
df_subgroup_summary = mdiag.subgroup_summary(df_edges_classified)
display(df_subgroup_summary)

df_subgroup_summary.to_csv(
    TABLES_DIR / "residual_subgroup_summary.csv", index=False,
)
print(f"Saved: {TABLES_DIR / 'residual_subgroup_summary.csv'}")

# Source x target sector flow matrix (non-managers)
flow_matrix = mdiag.non_manager_flow_matrix(
    df_edges_classified, df_soc_centres,
    column_normalise=True,
)
print("\nNon-manager flows: source-sector share by target-sector (%)")
display(flow_matrix)
flow_matrix.to_csv(TABLES_DIR / "residual_nonmgr_flow_matrix.csv")

In [ ]:
# Compute the synthesis (one arrow per subgroup)
df_synthesis = mdiag.flow_synthesis(df_edges_classified)

# Detailed subgroups figure
plots.plot_residual_subgroups(
    df_edges_classified=df_edges_classified,
    src_positions=src_positions,
    grid=grid,
    separatrix=separatrix,
    occupations_xy=(xo, yo),
    out_path=FIGURES_DIR / "residual_subgroups_gow.png",
    theme="paper",
)
print(f"Saved: {FIGURES_DIR / 'residual_subgroups_gow.png'}")

# Synthesis figure (one arrow per subgroup)
plots.plot_residual_synthesis(
    flow_synthesis_df=df_synthesis,
    df_edges_classified=df_edges_classified,
    grid=grid,
    separatrix=separatrix,
    occupations_xy=(xo, yo),
    out_path=FIGURES_DIR / "residual_synthesis_gow.png",
    theme="paper",
)
print(f"Saved: {FIGURES_DIR / 'residual_synthesis_gow.png'}")